# Interoperability with C: the `ctypes` module

This demonstration writes two C functions, the product and the cumulative sum of the values of an array, compiles them into a shared library, then calls them from Python with `ctypes`. Last, it compares the time of the cumulative sum in C with that of a Python loop and of `np.cumsum`, as the Cython demonstration does.

## Writing the C file

In [1]:
%%writefile functions.c
double prod(double *x, int n) {
  double prod = 1.0;
  for (int i = 0; i < n; i++) {
    prod *= x[i];
  }
  return prod;
}

void cumsum(double *a, double *b, int n) {
  if (n <= 0) {
    return;
  }
  b[0] = a[0];
  for (int i = 1; i < n; i++) {
    b[i] = a[i] + b[i - 1];
  }
}

Writing functions.c


## Compiling the shared library

In [2]:
!gcc -c -Wall -O2 -fPIC functions.c -o functions.o
!gcc -shared -o libfunctions.so functions.o

## Loading the library and declaring the functions' types

In [3]:
import ctypes
import pathlib

import numpy as np

libname = "./libfunctions.so"
if not pathlib.Path(libname).exists():
  raise FileNotFoundError(f"{libname} not found")

lib = ctypes.CDLL(libname)

lib.prod.argtypes = [ctypes.POINTER(ctypes.c_double), ctypes.c_int]
lib.prod.restype = ctypes.c_double

lib.cumsum.argtypes = [
  ctypes.POINTER(ctypes.c_double),
  ctypes.POINTER(ctypes.c_double),
  ctypes.c_int,
]
lib.cumsum.restype = None

## Python functions around the C functions

They convert their argument into a contiguous NumPy array of `float64`, the type C expects, and pass a pointer to its data.

In [4]:
def prod(x):
  arr = np.ascontiguousarray(x, dtype=np.float64)
  ptr = arr.ctypes.data_as(ctypes.POINTER(ctypes.c_double))
  return lib.prod(ptr, len(arr))


def cumsum(a):
  a = np.ascontiguousarray(a, dtype=np.float64)
  b = np.empty_like(a)
  ptr_a = a.ctypes.data_as(ctypes.POINTER(ctypes.c_double))
  ptr_b = b.ctypes.data_as(ctypes.POINTER(ctypes.c_double))
  lib.cumsum(ptr_a, ptr_b, len(a))
  return b

## Using the functions from Python

In [5]:
print(prod([1, 2, 3, 4]))
print(cumsum([1, 2, 3, 4]))
print(cumsum([]))

24.0
[ 1.  3.  6. 10.]
[]


## Timing

The cumulative sum of a million values, three ways: a Python loop, over a list, then the C function, then `np.cumsum`.

In [6]:
def cumsum_python(values):
  b = []
  total = 0.0
  for x in values:
    total += x
    b.append(total)
  return b


a = np.linspace(-1, 1, 1_000_000)
values = a.tolist()

In [7]:
%timeit cumsum_python(values)

79.8 ms ± 16.3 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [8]:
%timeit cumsum(a)

2.02 ms ± 78.5 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [9]:
%timeit np.cumsum(a)

4.18 ms ± 126 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


The C function is more than ten times faster than the Python loop, and at least as fast as `np.cumsum`, itself written in C.